# LUMI 정지 이미지 패턴 스크리닝 v2 (`image_discriminator_2.ipynb`)

`image_discriminator_1.ipynb`(v1)의 리뷰 결과를 반영한 개선판이다. 계산은 모두 `src/lumi/static_features.py`에 있고, 이 노트북은 실행·표시만 담당한다.

| 항목 | v1 | v2 |
|---|---|---|
| 줄무늬 쌍 계산 | `경계 수 // 2` → 항상 1쌍 누락 (5쌍 → 4쌍) | 줄무늬 수를 직접 셈 (정확히 5쌍 → 5쌍) |
| 패턴 면적 | 쌍 개수(중앙 선)와 면적(화면 전체 대비)을 따로 계산 | **5쌍 이상이 검출된 구간만** 면적에 포함, 인접 주사선 일관성 확인 |
| 탐색 범위 | 화면 중앙을 지나는 선만 | 화면 전체 가로·세로 주사선 (세로·가로·사선·동심원·방사형) |
| 고대비 / 고휘도 / 적색 | 정지 화면에 깜빡임 기준을 적용해 '위험' 판정 | 판정하지 않음 → 시간축 판정에 쓸 **특징값** |
| 확률 3종 | '확률', '환자 위험 확률' | '확률' 제거. Binnie 곡선은 시청 기하를 입력받는 **참고값** |
| 해상도 | 1080px로 축소 (가는 줄무늬 왜곡) | 원본 해상도 (4K 초과만 축소하고 기록) |
| 입력 | 고정 경로 `C:\LUMI\이미지\패턴.png` | `IMAGE_PATH` 지정, 없으면 합성 예제 세트 |

> **안전:** 이 노트북은 입력 이미지를 화면에 표시하지 않는다. 결과는 수치 표와 패턴 영역 마스크(흑백 면적) 파일뿐이다. 출력은 연구용 스크리닝 값이며 ITU-R BT.1702 / WCAG 적합성 판정이나 임상적 안전성 판단이 아니다.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import cv2
import numpy as np
import pandas as pd
from lumi.static_features import ViewingGeometry, analyze_image, load_image_rgb

IMAGE_PATH = None   # 예: r"C:\LUMI\이미지\패턴.png"  — None이면 합성 예제 세트를 분석
GEOMETRY = ViewingGeometry(diagonal_inch=27, distance_cm=60)   # 시청 환경: 27인치 모니터, 60cm
print("저장소:", ROOT)

저장소: C:\Users\baneu\Desktop\정호\고려대\지능정보아카데미_고려대(유길상교수님)\아이디어 발표(PSE)\lumi_lab_hamsungdo_research_v03


In [2]:
def demo_images(h: int = 540, w: int = 960) -> dict[str, np.ndarray]:
    """v1 리뷰에서 오판을 재현한 사례 + 실제 패턴 사례 (화면에 표시하지 않음)."""
    yy, xx = np.mgrid[:h, :w]
    gray = lambda g: np.repeat((g * 255).astype(np.uint8)[..., None], 3, axis=2)
    out = {}

    doc = np.full((h, w, 3), 255, np.uint8)
    words = "The quick brown fox jumps over the lazy dog LUMI research 2026".split()
    rng = np.random.default_rng(1)
    for y in range(36, h - 10, 26):
        x = 30
        while True:
            word = words[rng.integers(len(words))]
            (tw, _), _ = cv2.getTextSize(word, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
            if x + tw > w - 30:
                break
            cv2.putText(doc, word, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (20, 20, 20), 1, cv2.LINE_AA)
            x += tw + 10
    out["흰 배경 문서"] = doc

    noisy = np.full((h, w, 3), 255, np.uint8)
    for _ in range(1500):
        y, x = rng.integers(0, h - 6), rng.integers(0, w - 12)
        noisy[y:y + 5, x:x + rng.integers(3, 12)] = 0
    cy, cx = h // 2, w // 2
    noisy[cy - 45:cy + 45, cx - 90:cx + 90] = 255
    for i in range(0, 180, 12):
        noisy[cy - 45:cy + 45, cx - 90 + i:cx - 84 + i] = 0
    out["작은 줄무늬 조각 + 글자형 블록"] = noisy

    out["좌흑 / 우백"] = gray(xx >= w // 2)
    red = np.full((h, w, 3), 255, np.uint8); red[:, : int(w * 0.45)] = (255, 0, 0)
    out["포화 적색 45% (정지)"] = red
    out["세로 줄무늬 전체"] = gray((xx // 6) % 2)
    out["동심원 전체"] = gray((np.hypot(yy - h / 2, xx - w / 2) // 10) % 2)
    half = np.full((h, w, 3), 128, np.uint8)
    for x in range(0, w // 2, 10):
        half[:, x:x + 5] = 255; half[:, x + 5:x + 10] = 0
    out["화면 왼쪽 절반 줄무늬"] = half
    return out

images = {Path(IMAGE_PATH).name: load_image_rgb(IMAGE_PATH)} if IMAGE_PATH else demo_images()
print(f"분석 대상 {len(images)}개:", ", ".join(images))

분석 대상 7개: 흰 배경 문서, 작은 줄무늬 조각 + 글자형 블록, 좌흑 / 우백, 포화 적색 45% (정지), 세로 줄무늬 전체, 동심원 전체, 화면 왼쪽 절반 줄무늬


In [3]:
results, rows = {}, []
for name, img in images.items():
    r = analyze_image(img, geometry=GEOMETRY)
    results[name] = r
    p, f, ref = r["pattern"], r["features"], r["reference"]
    rows.append({
        "이미지": name,
        "최대 줄무늬 쌍": p["max_pairs"],
        "패턴 면적 %": round(p["area_fraction"] * 100, 1),
        "패턴 >25%": "위험" if p["pattern_over_25pct"] else "-",
        "패턴 >40%": "위험" if p["pattern_over_40pct"] else "-",
        "평균 휘도 cd/m²": round(f["mean_cd"], 1),
        "160cd/m² 이상 면적 %": round(f["bright_area_ge160"] * 100, 1),
        "포화 적색 면적 %": round(f["saturated_red_area"] * 100, 1),
        "패턴 시각도 °": round(ref["theta_deg"], 1),
        "Binnie 참고값": round(ref["binnie_reference"], 2),
    })
table = pd.DataFrame(rows).set_index("이미지")
table

,최대 줄무늬 쌍,패턴 면적 %,패턴 >25%,패턴 >40%,평균 휘도 cd/m²,160cd/m² 이상 면적 %,포화 적색 면적 %,패턴 시각도 °,Binnie 참고값
이미지,,,,,,,,,
흰 배경 문서,6,0.8,-,-,178.5,85.8,0.0,4.2,0.27
작은 줄무늬 조각 + 글자형 블록,17,3.5,-,-,178.1,89.1,0.0,9.0,0.66
좌흑 / 우백,0,0.0,-,-,100.0,50.0,0.0,0.0,0.00
포화 적색 45% (정지),0,0.0,-,-,129.1,55.0,45.0,0.0,0.00
세로 줄무늬 전체,80,100.0,위험,위험,100.0,50.0,0.0,45.7,1.00
동심원 전체,48,99.9,위험,위험,100.0,50.0,0.0,45.7,1.00
화면 왼쪽 절반 줄무늬,48,50.0,위험,위험,71.6,25.0,0.0,33.2,1.00


## 결과 읽는 법

- **판정은 '패턴 >25%', '패턴 >40%' 두 열뿐이다.** 조건: 같은 영역에서 명암 대비 규칙(ΔL≥20 cd/m², 어두운 쪽 ≥160이면 Michelson>1/17)을 만족하는 줄무늬가 **5쌍 이상**이고, 그런 영역의 면적이 기준을 넘을 때.
- **휘도·적색 열은 특징값**이다. 표준의 20 cd/m²·적색비 0.8·Δu′v′ 0.2는 프레임 사이 *변화*(깜빡임)의 조건이므로, 정지 이미지 한 장으로는 위험을 판정하지 않는다. 다음 단계의 시간축 판정기 입력으로 쓴다.
- **Binnie 참고값**은 패턴 자극에 대한 광과민 환자의 뇌파 반응 비율 곡선을 시청 기하에 대입한 값이다. 발작 확률이 아니다. θ를 '패턴 면적과 같은 넓이의 원 지름 시각도'로 해석한 것은 가정이며, Jordan & Vanderheiden (2024) 부록 A 원문과 대조가 필요하다.
- **보수적 계수:** 패턴을 둘러싼 넓은 바탕도 경계 줄무늬 1개로 센다. 그래서 회색 바탕 위 4쌍은 5쌍으로 셀 수 있다(과소 계수보다 과대 계수를 택함).

In [4]:
# 패턴 영역 마스크만 저장 (입력 이미지는 저장·표시하지 않음)
mask_dir = ROOT / "output" / "static_masks"
mask_dir.mkdir(parents=True, exist_ok=True)
for i, (name, r) in enumerate(results.items()):
    path = mask_dir / f"{i:02d}_mask.png"
    cv2.imencode(".png", r["mask"].astype(np.uint8) * 255)[1].tofile(str(path))
    print(f"{path.name}: {name} — 패턴 면적 {r['pattern']['area_fraction'] * 100:.1f}%")

00_mask.png: 흰 배경 문서 — 패턴 면적 0.8%
01_mask.png: 작은 줄무늬 조각 + 글자형 블록 — 패턴 면적 3.5%
02_mask.png: 좌흑 / 우백 — 패턴 면적 0.0%
03_mask.png: 포화 적색 45% (정지) — 패턴 면적 0.0%
04_mask.png: 세로 줄무늬 전체 — 패턴 면적 100.0%
05_mask.png: 동심원 전체 — 패턴 면적 99.9%
06_mask.png: 화면 왼쪽 절반 줄무늬 — 패턴 면적 50.0%


## 한계와 다음 단계

1. 주사선은 가로·세로 두 방향이다. 줄무늬가 정확히 45°로 매우 가늘면(1px 미만 간격) 표본화 한계로 놓칠 수 있다.
2. 휘도는 sRGB·흰색 200 cd/m² 가정이다. 실제 표시 장치 밝기·HDR은 반영하지 않는다.
3. 패턴 면적 기준 25%/40%는 v1의 설정을 그대로 썼다. 적용 표준·정책 버전을 결정해 `docs/EXPERIMENT_PLAN.md`에 기록해야 한다.
4. 다음 단계: 이 특징값을 프레임마다 계산해 **프레임 간 휘도·적색 변화(깜빡임)** 판정기로 확장한다.
5. 회귀 테스트: `tests/test_static_features.py` (v1 오판 사례 포함).